# WordLab: CBOW and Skip-gram accuracy in Google Colab

This notebook trains both models using the project's existing training code and bundled corpus, then displays their training and held-out test accuracy. It does not change the main app or backend.

## Before running

In Google Colab, upload this notebook and select **both** `word2vec_engine.py` and `sample_corpus.txt` when prompted. In a local Jupyter notebook, open it from the project folder; it will use those files directly. Run the cells from top to bottom. The **held-out test accuracy** is the better estimate of performance on unseen sentences; training accuracy is measured on text the model has already seen.

In [1]:
%pip -q install "gensim>=4.3.3,<5" "numpy>=1.26,<3"

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path

required_files = {"word2vec_engine.py", "sample_corpus.txt"}
if not all(Path(name).is_file() for name in required_files):
    try:
        from google.colab import files
    except ImportError as error:
        raise FileNotFoundError(
            "Run this notebook from the project folder, or upload word2vec_engine.py "
            "and sample_corpus.txt in Google Colab."
        ) from error
    uploaded = files.upload()
    missing_files = required_files - set(uploaded)
    if missing_files:
        raise FileNotFoundError(
            "Please upload both project files: " + ", ".join(sorted(missing_files))
        )

corpus_text = Path("sample_corpus.txt").read_text(encoding="utf-8")
print("Loaded bundled Alice's Adventures in Wonderland sample.")

Loaded bundled Alice's Adventures in Wonderland sample.


In [3]:
from word2vec_engine import train_word2vec
import pandas as pd

# These settings match the main app.
settings = {
    "vector_size": 80,
    "window": 3,
    "epochs": 60,
    "test_fraction": 0.2,
    "min_count": 1,
    "seed": 42,
}

results = {}
for architecture in ("CBOW", "Skip-gram"):
    print(f"Training {architecture}...")
    results[architecture] = train_word2vec(
        corpus_text,
        architecture=architecture,
        **settings,
    )

print("Both models have finished training and evaluation.")

Training CBOW...
Training Skip-gram...
Both models have finished training and evaluation.


In [4]:
rows = []
for architecture, result in results.items():
    rows.append({
        "Model": architecture,
        "Training Top-1 (%)": round(result.training_metrics["top1_accuracy"] * 100, 1),
        "Training Top-5 / Hit@5 (%)": round(result.training_metrics["top5_accuracy"] * 100, 1),
        "Held-out Test Top-1 / Hit@1 (%)": round(result.metrics["top1_accuracy"] * 100, 1),
        "Held-out Test Top-5 / Hit@5 (%)": round(result.metrics["top5_accuracy"] * 100, 1),
        "Test examples scored": result.metrics["evaluated_tokens"],
    })

display(pd.DataFrame(rows).set_index("Model"))
for architecture, result in results.items():
    print(
        f"{architecture}: {result.train_sentence_count} total sentences; "
        f"{result.test_sentence_count} held out for testing; "
        f"{result.vocabulary_size} words in the full-corpus model."
    )

,Training Top-1 (%),Training Top-5 / Hit@5 (%),Held-out Test Top-1 / Hit@1 (%),Held-out Test Top-5 / Hit@5 (%),Test examples scored
Model,,,,,
CBOW,11.1,23.4,6.7,14.4,721
Skip-gram,13.3,44.9,6.3,19.6,682


CBOW: 405 total sentences; 81 held out for testing; 1494 words in the full-corpus model.
Skip-gram: 405 total sentences; 81 held out for testing; 1494 words in the full-corpus model.


For the project report, use the **Held-out Test** columns when describing performance on unseen sentences. Top-1 is the percentage of predictions where the correct answer is ranked first; Top-5 / Hit@5 checks whether it appears among the first five predictions. Skip-gram's scores measure whether actual neighboring words appear in its predictions.